<a href="https://colab.research.google.com/github/Rayudu-Somisetty/deep_learning_lab_tasks/blob/main/sigmoid_func_for_cancer_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# sigmoid function for cancer dataset
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# ===============================
# LOAD & CLEAN DATASET
# ===============================
data = pd.read_csv("data.csv")
if 'id' in data.columns:
    data = data.drop(columns=['id'])
data['diagnosis'] = data['diagnosis'].map({'B': 0, 'M': 1})

# CLEANING (fixes warnings!)
data = data.replace('?', np.nan).fillna(0).dropna()
var_scores = data.drop('diagnosis', axis=1).var()
low_var_cols = var_scores[var_scores < 1e-8].index
print(f"Dropped {len(low_var_cols)} zero-var cols")
data = data.drop(columns=low_var_cols)
data = data.apply(pd.to_numeric, errors='coerce').fillna(0)

categorical_cols = data.select_dtypes(include=['object']).columns
data = pd.get_dummies(data, columns=categorical_cols, drop_first=True)

X = data.drop(columns=['diagnosis']).values
y = data['diagnosis'].values
print(f"Clean shape: {X.shape}, Classes: {np.bincount(y)}")

# ===============================
# BETTER SPLIT
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y  # 75/25 balanced
)

# ===============================
# SCALING (now safe)
# ===============================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ===============================
# SAFE SIGMOID
# ===============================
def sigmoid(x):
    x = np.clip(x, -250, 250)  # Prevent overflow -> NaN
    return 1 / (1 + np.exp(-x))

# ===============================
# FIXED MODEL (batch GD)
# ===============================
class SigmoidClassifier:
    def __init__(self, lr=0.001, ite=5000):  # Smaller lr!
        self.lr = lr
        self.ite = ite

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1]) # (n_features,)
        self.b = 0
        m = len(y)
        # y should be (m,) for direct element-wise subtraction with y_pred
        # y = y.reshape(-1, 1) # REMOVED: This was causing y to be (m,1)

        for i in range(self.ite):
            z = np.dot(X, self.w) + self.b # z has shape (m,)
            y_pred = sigmoid(z) # y_pred has shape (m,)

            error = y_pred - y # Now error will have shape (m,)

            # dw = (1/m) * np.dot(X.T, error) => X.T (n_features,m) @ error (m,) => dw (n_features,)
            dw = (1/m) * np.dot(X.T, error) # dw has correct shape (n_features,)
            db = (1/m) * np.sum(error) # db is scalar

            self.w -= self.lr * dw # dw is already 1D and correctly shaped
            self.b -= self.lr * db

    def predict(self, X):
        z = np.dot(X, self.w) + self.b
        return (sigmoid(z) >= 0.5).astype(int).flatten()

# ===============================
# TRAIN & TEST
# ===============================
model = SigmoidClassifier(lr=0.001, ite=5000)
model.fit(X_train, y_train)

print("Sigmoid Train Acc:", accuracy_score(y_train, model.predict(X_train)))
print("Sigmoid Test Acc :", accuracy_score(y_test, model.predict(X_test)))

Dropped 1 zero-var cols
Clean shape: (569, 30), Classes: [357 212]
Sigmoid Train Acc: 0.971830985915493
Sigmoid Test Acc : 0.972027972027972
